####Ex 1→ Setup - Create all 4 DataFrames
Creates 4 small DataFrames that all other exercises use: customers (8 rows), products (6 rows), orders (12 rows), discounts (5 rows). Intentionally designed with missing matches, nulls, and edge cases so every join type produces a meaningful result.

In [0]:
# ── Run this cell FIRST — all exercises depend on these DataFrames ──

from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
from pyspark.sql.functions import col, round as spark_round, broadcast, coalesce, lit

# ── customers (8 rows) ───────────────────────────────────────
customers = spark.createDataFrame([
    (1, "Alice",   "Delhi",     "Gold"),
    (2, "Bob",     "Mumbai",    "Silver"),
    (3, "Charlie", "Bangalore", "Gold"),
    (4, "Diana",   "Delhi",     "Bronze"),
    (5, "Eve",     "Mumbai",    "Silver"),
    (6, "Frank",   "Chennai",   "Gold"),
    (7, "Grace",   "Bangalore", "Bronze"),
    (8, "Henry",   "Delhi",     "Silver"),
], ["cust_id", "name", "city", "tier"])

# ── products (6 rows) ────────────────────────────────────────
products = spark.createDataFrame([
    (101, "Laptop",     "Electronics", 75000.0),
    (102, "Phone",      "Electronics", 25000.0),
    (103, "T-Shirt",    "Clothing",     999.0),
    (104, "Jeans",      "Clothing",    2499.0),
    (105, "Rice 5kg",   "Grocery",      450.0),
    (106, "Headphones", "Electronics", 3500.0),
], ["prod_id", "prod_name", "category", "price"])

# ── orders (12 rows — some customers have no orders, some products never ordered) ──
orders = spark.createDataFrame([
    (1001, 1, 101, 1, "2024-01-10", "Delivered"),
    (1002, 2, 102, 2, "2024-01-15", "Delivered"),
    (1003, 1, 103, 3, "2024-01-20", "Delivered"),
    (1004, 3, 101, 1, "2024-02-01", "Delivered"),
    (1005, 4, 105, 5, "2024-02-10", "Cancelled"),
    (1006, 2, 104, 1, "2024-02-14", "Delivered"),
    (1007, 5, 102, 1, "2024-03-01", "Delivered"),
    (1008, 3, 106, 2, "2024-03-05", "Delivered"),
    (1009, 1, 104, 2, "2024-03-10", "Returned"),
    (1010, 6, 101, 1, "2024-03-15", "Delivered"),
    (1011, 7, 105, 3, "2024-03-20", "Delivered"),
    (1012, 2, 106, 1, "2024-03-25", "Delivered"),
], ["order_id", "cust_id", "prod_id", "qty", "order_date", "status"])

# ── discount_codes (only for Gold/Silver tiers, not all products) ──
discounts = spark.createDataFrame([
    ("Gold",   101, 10.0),
    ("Gold",   102,  8.0),
    ("Gold",   106, 12.0),
    ("Silver", 102,  5.0),
    ("Silver", 104,  3.0),
], ["tier", "prod_id", "discount_pct"])

print("customers:", customers.count(), "| products:", products.count(),
      "| orders:", orders.count(), "| discounts:", discounts.count())
customers.display()
products.display()
orders.display()
discounts.display()

customers: 8 | products: 6 | orders: 12 | discounts: 5


cust_id,name,city,tier
1,Alice,Delhi,Gold
2,Bob,Mumbai,Silver
3,Charlie,Bangalore,Gold
4,Diana,Delhi,Bronze
5,Eve,Mumbai,Silver
6,Frank,Chennai,Gold
7,Grace,Bangalore,Bronze
8,Henry,Delhi,Silver


prod_id,prod_name,category,price
101,Laptop,Electronics,75000.0
102,Phone,Electronics,25000.0
103,T-Shirt,Clothing,999.0
104,Jeans,Clothing,2499.0
105,Rice 5kg,Grocery,450.0
106,Headphones,Electronics,3500.0


order_id,cust_id,prod_id,qty,order_date,status
1001,1,101,1,2024-01-10,Delivered
1002,2,102,2,2024-01-15,Delivered
1003,1,103,3,2024-01-20,Delivered
1004,3,101,1,2024-02-01,Delivered
1005,4,105,5,2024-02-10,Cancelled
1006,2,104,1,2024-02-14,Delivered
1007,5,102,1,2024-03-01,Delivered
1008,3,106,2,2024-03-05,Delivered
1009,1,104,2,2024-03-10,Returned
1010,6,101,1,2024-03-15,Delivered


tier,prod_id,discount_pct
Gold,101,10.0
Gold,102,8.0
Gold,106,12.0
Silver,102,5.0
Silver,104,3.0


####Ex 2→ INNER JOIN - only matching rows
INNER JOIN returns only rows that have a match in BOTH tables. If a customer has no orders, they disappear. If an order has no matching customer, it disappears too. The most common join type.

In [0]:
# INNER JOIN
  # orders with customers
print("Checking both tables (orders and customers) for getting reference for joining point...")
print("orders table...")
orders.display()
print("customers table...")
customers.display()

# Doing joins on both table
result = orders.join(customers, on="cust_id", how="inner")
print("orders with customers INNER JOIN Table...")
result.display()

# Add products table too - 3 table inner joins
print("Checking products table for getting reference for joining point with previous 2 tables...")
products.display()

# Doing joins on 3 tables
print("orders INNER JOIN table with customers and products...")
full = orders.join(customers, on="cust_id", how="inner") \
             .join(products, on="prod_id" , how="inner")

full.display()


Checking both tables (orders and customers) for getting reference for joining point...
orders table...


order_id,cust_id,prod_id,qty,order_date,status
1001,1,101,1,2024-01-10,Delivered
1002,2,102,2,2024-01-15,Delivered
1003,1,103,3,2024-01-20,Delivered
1004,3,101,1,2024-02-01,Delivered
1005,4,105,5,2024-02-10,Cancelled
1006,2,104,1,2024-02-14,Delivered
1007,5,102,1,2024-03-01,Delivered
1008,3,106,2,2024-03-05,Delivered
1009,1,104,2,2024-03-10,Returned
1010,6,101,1,2024-03-15,Delivered


customers table...


cust_id,name,city,tier
1,Alice,Delhi,Gold
2,Bob,Mumbai,Silver
3,Charlie,Bangalore,Gold
4,Diana,Delhi,Bronze
5,Eve,Mumbai,Silver
6,Frank,Chennai,Gold
7,Grace,Bangalore,Bronze
8,Henry,Delhi,Silver


orders with customers INNER JOIN Table...


cust_id,order_id,prod_id,qty,order_date,status,name,city,tier
1,1001,101,1,2024-01-10,Delivered,Alice,Delhi,Gold
2,1002,102,2,2024-01-15,Delivered,Bob,Mumbai,Silver
1,1003,103,3,2024-01-20,Delivered,Alice,Delhi,Gold
3,1004,101,1,2024-02-01,Delivered,Charlie,Bangalore,Gold
4,1005,105,5,2024-02-10,Cancelled,Diana,Delhi,Bronze
2,1006,104,1,2024-02-14,Delivered,Bob,Mumbai,Silver
5,1007,102,1,2024-03-01,Delivered,Eve,Mumbai,Silver
3,1008,106,2,2024-03-05,Delivered,Charlie,Bangalore,Gold
1,1009,104,2,2024-03-10,Returned,Alice,Delhi,Gold
6,1010,101,1,2024-03-15,Delivered,Frank,Chennai,Gold


Checking products table for getting reference for joining point with previous 2 tables...


prod_id,prod_name,category,price
101,Laptop,Electronics,75000.0
102,Phone,Electronics,25000.0
103,T-Shirt,Clothing,999.0
104,Jeans,Clothing,2499.0
105,Rice 5kg,Grocery,450.0
106,Headphones,Electronics,3500.0


orders INNER JOIN table with customers and products...


prod_id,cust_id,order_id,qty,order_date,status,name,city,tier,prod_name,category,price
101,1,1001,1,2024-01-10,Delivered,Alice,Delhi,Gold,Laptop,Electronics,75000.0
102,2,1002,2,2024-01-15,Delivered,Bob,Mumbai,Silver,Phone,Electronics,25000.0
103,1,1003,3,2024-01-20,Delivered,Alice,Delhi,Gold,T-Shirt,Clothing,999.0
101,3,1004,1,2024-02-01,Delivered,Charlie,Bangalore,Gold,Laptop,Electronics,75000.0
105,4,1005,5,2024-02-10,Cancelled,Diana,Delhi,Bronze,Rice 5kg,Grocery,450.0
104,2,1006,1,2024-02-14,Delivered,Bob,Mumbai,Silver,Jeans,Clothing,2499.0
102,5,1007,1,2024-03-01,Delivered,Eve,Mumbai,Silver,Phone,Electronics,25000.0
106,3,1008,2,2024-03-05,Delivered,Charlie,Bangalore,Gold,Headphones,Electronics,3500.0
104,1,1009,2,2024-03-10,Returned,Alice,Delhi,Gold,Jeans,Clothing,2499.0
101,6,1010,1,2024-03-15,Delivered,Frank,Chennai,Gold,Laptop,Electronics,75000.0


####Ex 3→ JOIN on multiple columns - composite key
Sometimes one column isn't enough to uniquely identify a match. Use a list of columns or an expression for composite key joins — common when joining on (customer_id + product_id) or (date + region).

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Join orders with discounts on BOTH tier and prod_id
# A discount only applies if the customer's tier and product both match

orders_with_tier = orders.join(customers.select("cust_id","name","tier"), on="cust_id",how="inner")
orders_with_tier.display()

# Method 1: list of column names (works when column names are the same)
discounts.display()

with_discount = orders_with_tier.join(
    discounts,
    on=["tier","prod_id"], # composite key -both must match
    how="left"
)

with_discount.select(
    "order_id", "name", "tier", "prod_id", "qty", "discount_pct"
).orderBy("order_id").display()

# Method 2: explicit expression (use when column names differ)
with_discount2 = orders_with_tier.join(
    discounts,
    on=(orders_with_tier["tier"] == discounts["tier"]) &
       (orders_with_tier["prod_id"] == discounts["prod_id"]),
    how="left"
)

with_discount2.display()

# Count how many orders got a discount vs none
with_discount.withColumn("got_discount",
              when(col("discount_pct").isNotNull(), "Yes").otherwise("No")
).groupBy("got_discount").count().display()


cust_id,order_id,prod_id,qty,order_date,status,name,tier
1,1001,101,1,2024-01-10,Delivered,Alice,Gold
2,1002,102,2,2024-01-15,Delivered,Bob,Silver
1,1003,103,3,2024-01-20,Delivered,Alice,Gold
3,1004,101,1,2024-02-01,Delivered,Charlie,Gold
4,1005,105,5,2024-02-10,Cancelled,Diana,Bronze
2,1006,104,1,2024-02-14,Delivered,Bob,Silver
5,1007,102,1,2024-03-01,Delivered,Eve,Silver
3,1008,106,2,2024-03-05,Delivered,Charlie,Gold
1,1009,104,2,2024-03-10,Returned,Alice,Gold
6,1010,101,1,2024-03-15,Delivered,Frank,Gold


tier,prod_id,discount_pct
Gold,101,10.0
Gold,102,8.0
Gold,106,12.0
Silver,102,5.0
Silver,104,3.0


order_id,name,tier,prod_id,qty,discount_pct
1001,Alice,Gold,101,1,10.0
1002,Bob,Silver,102,2,5.0
1003,Alice,Gold,103,3,null
1004,Charlie,Gold,101,1,10.0
1005,Diana,Bronze,105,5,null
1006,Bob,Silver,104,1,3.0
1007,Eve,Silver,102,1,5.0
1008,Charlie,Gold,106,2,12.0
1009,Alice,Gold,104,2,null
1010,Frank,Gold,101,1,10.0


cust_id,order_id,prod_id,qty,order_date,status,name,tier,tier,prod_id,discount_pct
1,1001,101,1,2024-01-10,Delivered,Alice,Gold,Gold,101,10.0
2,1002,102,2,2024-01-15,Delivered,Bob,Silver,Silver,102,5.0
1,1003,103,3,2024-01-20,Delivered,Alice,Gold,null,null,null
3,1004,101,1,2024-02-01,Delivered,Charlie,Gold,Gold,101,10.0
4,1005,105,5,2024-02-10,Cancelled,Diana,Bronze,null,null,null
2,1006,104,1,2024-02-14,Delivered,Bob,Silver,Silver,104,3.0
5,1007,102,1,2024-03-01,Delivered,Eve,Silver,Silver,102,5.0
3,1008,106,2,2024-03-05,Delivered,Charlie,Gold,Gold,106,12.0
1,1009,104,2,2024-03-10,Returned,Alice,Gold,null,null,null
6,1010,101,1,2024-03-15,Delivered,Frank,Gold,Gold,101,10.0


got_discount,count
Yes,7
No,5


####Ex 4→ LEFT JOIN - keep all rows from the left table
LEFT JOIN keeps every row from the left table. If no match is found in the right table, the right columns are filled with null. Use this when you want to enrich data but still keep records that have no match.

In [0]:
# LEFT JOIN: all customers, with their orders if they exist
# Customer 8 (Henry) has no orders - he should appear with nulls

cust_orders = customers.join(orders, on="cust_id", how="left")
cust_orders.display()

# Find customers with No orders (null order_id after left join)
no_orders = customers.join(orders, on="cust_id", how="left") \
                     .filter(col("order_id").isNull()) \
                     .select("cust_id","name","city","tier")

no_orders.display()

cust_id,name,city,tier,order_id,prod_id,qty,order_date,status
1,Alice,Delhi,Gold,1009,104,2,2024-03-10,Returned
1,Alice,Delhi,Gold,1003,103,3,2024-01-20,Delivered
1,Alice,Delhi,Gold,1001,101,1,2024-01-10,Delivered
2,Bob,Mumbai,Silver,1012,106,1,2024-03-25,Delivered
2,Bob,Mumbai,Silver,1006,104,1,2024-02-14,Delivered
2,Bob,Mumbai,Silver,1002,102,2,2024-01-15,Delivered
3,Charlie,Bangalore,Gold,1008,106,2,2024-03-05,Delivered
3,Charlie,Bangalore,Gold,1004,101,1,2024-02-01,Delivered
4,Diana,Delhi,Bronze,1005,105,5,2024-02-10,Cancelled
5,Eve,Mumbai,Silver,1007,102,1,2024-03-01,Delivered


cust_id,name,city,tier
8,Henry,Delhi,Silver


####Ex 5→ RIGHT JOIN and FULL OUTER JOIN
RIGHT JOIN keeps all right-table rows. FULL OUTER JOIN keeps all rows from both tables — unmatched rows get nulls on the missing side. FULL OUTER is used for reconciliation and finding mismatches in both directions.

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# RIGHT JOIN: all products, with orders if they exist

prod_orders = products.join(orders, on="prod_id", how="right")
prod_orders.display()

# FULL OUTER JOIN: see unmatched rows from both sides
  # use case: reconsile two systems
# Simulate: system_a has some orders, system_b has some orders
system_a = orders.filter(col("order_id") <= 1006) \
                 .select(col("order_id"),col("cust_id").alias("cust_a"),col("status").alias("status_a"))

system_b = orders.filter(col("order_id") >= 1004) \
                 .select(col("order_id"), col("cust_id").alias("cust_b"), col("status").alias("status_b"))

reconciled = system_a.join(system_b, on="order_id", how="full")
reconciled.display()

reconciled.withColumn("match_status",
    when(col("cust_a").isNull(),  "Only in B")
    .when(col("cust_b").isNull(),  "Only in A")
    .otherwise("In both")
).display()

prod_id,prod_name,category,price,order_id,cust_id,qty,order_date,status
101,Laptop,Electronics,75000.0,1001,1,1,2024-01-10,Delivered
102,Phone,Electronics,25000.0,1002,2,2,2024-01-15,Delivered
103,T-Shirt,Clothing,999.0,1003,1,3,2024-01-20,Delivered
101,Laptop,Electronics,75000.0,1004,3,1,2024-02-01,Delivered
105,Rice 5kg,Grocery,450.0,1005,4,5,2024-02-10,Cancelled
104,Jeans,Clothing,2499.0,1006,2,1,2024-02-14,Delivered
102,Phone,Electronics,25000.0,1007,5,1,2024-03-01,Delivered
106,Headphones,Electronics,3500.0,1008,3,2,2024-03-05,Delivered
104,Jeans,Clothing,2499.0,1009,1,2,2024-03-10,Returned
101,Laptop,Electronics,75000.0,1010,6,1,2024-03-15,Delivered


order_id,cust_a,status_a,cust_b,status_b
1004,3,Delivered,3,Delivered
1005,4,Cancelled,4,Cancelled
1006,2,Delivered,2,Delivered
1007,null,null,5,Delivered
1008,null,null,3,Delivered
1009,null,null,1,Returned
1010,null,null,6,Delivered
1012,null,null,2,Delivered
1011,null,null,7,Delivered
1001,1,Delivered,null,null


order_id,cust_a,status_a,cust_b,status_b,match_status
1004,3,Delivered,3,Delivered,In both
1005,4,Cancelled,4,Cancelled,In both
1006,2,Delivered,2,Delivered,In both
1007,null,null,5,Delivered,Only in B
1008,null,null,3,Delivered,Only in B
1009,null,null,1,Returned,Only in B
1010,null,null,6,Delivered,Only in B
1012,null,null,2,Delivered,Only in B
1011,null,null,7,Delivered,Only in B
1001,1,Delivered,null,null,Only in A


####Ex 6→ LEFT ANTI JOIN - find records with no match
LEFT ANTI returns rows from the left table that have NO match in the right table. It is the most efficient way to implement NOT IN or NOT EXISTS in PySpark — and it is safer than NOT IN with nulls.

In [0]:
# FIND CUSTOMERS WHO HAVE NEVER PLACED ANY ORDER
no_order_customers = customers.join(orders, on="cust_id", how="left_anti")
print("CUSTOMERS WITH NO ORDERS:")
no_order_customers.display()

# FIND PRODUCTS THAT HAVE NEVER BEEN ORDERED
no_product_ordered = products.join(orders, on="prod_id", how="left_anti")
print("PRODUCTS WITH NO ORDERS:")
no_product_ordered.display()

# FIND ORDERS WITH STATUS NOT IN A DELIVERED WHITELIST
delivered_ids = orders.filter(col("status").contains("Delivered")) \
                      .select("order_id")
delivered_ids.display()

problem_orders = orders.join(delivered_ids, on="order_id", how="left_anti")
print("NON DELIVERED ORDERS LIST: ")
problem_orders.display()



CUSTOMERS WITH NO ORDERS:


cust_id,name,city,tier
8,Henry,Delhi,Silver


PRODUCTS WITH NO ORDERS:


prod_id,prod_name,category,price


order_id
1001
1002
1003
1004
1006
1007
1008
1010
1011
1012


NON DELIVERED ORDERS LIST: 


order_id,cust_id,prod_id,qty,order_date,status
1005,4,105,5,2024-02-10,Cancelled
1009,1,104,2,2024-03-10,Returned


####Ex 7→ LEFT SEMI JOIN - filter by existence
LEFT SEMI returns rows from the left table that DO have a match in the right — but unlike inner join it does NOT add the right table's columns. It is the efficient equivalent of WHERE EXISTS or WHERE id IN (...).

In [0]:
# FIND CUSTOMERS WHO HAVE ORDERED ATLEAST ONE ORDER
active_customers = customers.join(orders, on="cust_id", how="left_semi")
print("CUSTOMERS ORDERED AT LEAST ONCE: ")
active_customers.display()

# FIND PRODUCTS THAT WERE ORDERED AT LEAST ONCE
ordered_products = orders.join(products, on="prod_id", how="left_semi")
print("PRODUCTS ORDERED AT LEAST ONCE: ")
ordered_products.display()

# Practical use: filter a large customer table to only those
# who bought from a specific category (Electronics)
electronics_orders = orders.join(products.filter(col("category") == "Electronics"),on="prod_id", how="left_semi"
).select("cust_id").distinct()

electronics_buyers = customers.join(
    electronics_orders, on="cust_id", how="left_semi"
)

print("Customers who bought Electronics:")
electronics_buyers.display()

CUSTOMERS ORDERED AT LEAST ONCE: 


cust_id,name,city,tier
1,Alice,Delhi,Gold
2,Bob,Mumbai,Silver
3,Charlie,Bangalore,Gold
4,Diana,Delhi,Bronze
5,Eve,Mumbai,Silver
6,Frank,Chennai,Gold
7,Grace,Bangalore,Bronze


PRODUCTS ORDERED AT LEAST ONCE: 


prod_id,order_id,cust_id,qty,order_date,status
101,1001,1,1,2024-01-10,Delivered
102,1002,2,2,2024-01-15,Delivered
103,1003,1,3,2024-01-20,Delivered
101,1004,3,1,2024-02-01,Delivered
105,1005,4,5,2024-02-10,Cancelled
104,1006,2,1,2024-02-14,Delivered
102,1007,5,1,2024-03-01,Delivered
106,1008,3,2,2024-03-05,Delivered
104,1009,1,2,2024-03-10,Returned
101,1010,6,1,2024-03-15,Delivered


Customers who bought Electronics:


cust_id,name,city,tier
1,Alice,Delhi,Gold
2,Bob,Mumbai,Silver
3,Charlie,Bangalore,Gold
5,Eve,Mumbai,Silver
6,Frank,Chennai,Gold


####EX 8→ BROADCAST JOIN - small table optimisation
When one table is small (under a few MB), wrap it in broadcast() to send a full copy to every executor — eliminating the shuffle entirely. This is the single most impactful join optimisation in PySpark.

In [0]:
from pyspark.sql.functions import broadcast

# Without broadcast hint
result_normal = orders.join(products, on="prod_id", how="inner")
result_normal.explain()

# With broadcast hint
result_broadcast = orders.join(
    broadcast(products),
    on="prod_id",
    how="inner"
)

result_broadcast.explain()

print("=== WITHOUT broadcast ===")
orders.join(products, on="prod_id").explain()

print("\n=== WITH broadcast ===")
orders.join(
    broadcast(products),
    on="prod_id"
).explain()

# Practical enrichment
enriched = (
    orders
    .join(broadcast(products), on="prod_id", how="inner")
    .join(broadcast(customers), on="cust_id", how="inner")
    .select(
        "order_id",
        "name",
        "tier",
        "prod_name",
        "category",
        "price",
        "qty",
        spark_round(col("price") * col("qty"), 2).alias("order_value"),
        "status"
    )
)

enriched.show(truncate=False)

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- == Initial Plan ==
   PhotonResultStage
   +- PhotonColumnarToRow
      +- PhotonProject [prod_id#14768L, order_id#14766L, cust_id#14767L, qty#14769L, order_date#14770, status#14771, prod_name#14773, category#14774, price#14775]
         +- PhotonBroadcastHashJoin [prod_id#14768L], [prod_id#14772L], Inner, BuildRight, false, true
            :- PhotonRowToColumnar
            :  +- LocalTableScan [order_id#14766L, cust_id#14767L, prod_id#14768L, qty#14769L, order_date#14770, status#14771]
            +- PhotonShuffleExchangeSource
               +- PhotonShuffleMapStage EXECUTOR_BROADCAST, [id=#12047]
                  +- PhotonShuffleExchangeSink SinglePartition
                     +- PhotonRowToColumnar
                        +- LocalTableScan [prod_id#14772L, prod_name#14773, category#14774, price#14775]


== Photon Explanation ==
The query is fully supported by Photon.
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=fal